In [1]:
import torch
import os
import json
import numpy as np
import pandas as pd
import random
import statistics
from itertools import groupby
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib
from plotting_style import *
from risk_control_utils import (get_label_order, get_all_confidences, get_all_accuracies, get_ground_truth_by_type, 
                                get_relative_labels, apply_risk_control, load_all_data, get_losses_and_exits_confidence)

# Parameters
Specify all parameters for this notebook. 

In [3]:
use_calibration = True # True or False
fake_labels = True # True or False
confidence_type = 'argmax' # argmax or top2_diff or entropy
i_prop, c_prop = 50, 50 # the % of total observations of each type. do 5/95, 10/90, 25/75, 50/50
debug_mode = False # True to display small subset of plots in notebook, False to make all plots and save them
display_legends = False # whether to include legends on plots (all or none)
max_eps = 0.5 # the maximum epsilon-value to plot. For the paper, this is between 0.1 and 0.5. 

# Define Variables
These should not need to be modified to reproduce the results from the "Safe In-Context Learning" paper.

In [5]:
# Define a list of all the models and datasets to plot
models = ["facebook/layerskip-llama3-8B", "facebook/layerskip-llama2-7B", "meta-llama/Meta-Llama-3-8B", "meta-llama/Llama-2-7B-hf", "allenai/Olmo-3-1125-32B", "allenai/Olmo-3-1025-7B", "Qwen/Qwen3-8B"]
tokenizers = ["meta-llama/Meta-Llama-3-8B", "meta-llama/Llama-2-7B-hf", "meta-llama/Meta-Llama-3-8B", "meta-llama/Llama-2-7B-hf", "allenai/Olmo-3-1125-32B", "allenai/Olmo-3-1025-7B", "Qwen/Qwen3-8B"]
n_early_exits = [32, 32, 32, 32, 64, 32, 36]
datasets = ['sst2', 'trec', 'financial_phrasebank', 'tweeteval_hate', 'tweeteval_feminist', 'tweeteval_atheism', 'unnatural', 'ag_news']
n_demos=60 # 60

In [6]:
dataset_names_plot_titles = {
    'sst2': 'SST2',
    'trec': 'TREC',
    'financial_phrasebank': 'FinancialPhrasebank',
    'tweeteval_hate': 'TweetEval-Hate',
    'tweeteval_feminist': 'TweetEval-Feminist', 
    'tweeteval_atheism': 'TweetEval-Atheism',
    'unnatural': 'Unnatural', 
    'ag_news': 'AG News',
}

In [7]:
# Define base data folder to read from (and associated params for finding the right data files)
# different for different splits, e.g. 10_90_mix, 5_95_mix
mixed_filename = str(i_prop) + '_' + str(c_prop) + '_mix'
precomputed_risk_path = './rc-precomputed/' + ('fake_labels' if fake_labels else '') + ('/calibrated/' if use_calibration else '/uncalibrated/')
precomputed_risk_path += confidence_type + '/'
precomputed_mixed_path = precomputed_risk_path + mixed_filename + '/'
results_folder = './results' + ('_fake_labels' if fake_labels else '') + ('/calibrated' if use_calibration else '/uncalibrated')

with open('fake_labels.json') as f:
    fake_label_map = json.load(f)

In [8]:
# Define plotting params
c_color, i_color, z_color = 'tab:blue', 'tab:orange', 'tab:green'
model_colors = {"facebook/layerskip-llama3-8B": 'tab:brown', "facebook/layerskip-llama2-7B": 'tab:pink', "meta-llama/Meta-Llama-3-8B": 'tab:purple', 
               "meta-llama/Llama-2-7B-hf": 'tab:cyan', "allenai/Olmo-3-1125-32B": 'tab:gray', "allenai/Olmo-3-1025-7B": 'tab:olive',
               "Qwen/Qwen3-8B": 'tab:blue'}
# for risk control, this sets the earliest layer at which we are allowed to exit is 1/2 way through the total layers. 
plot_directory = './icl_plots' + ('_fake_labels' if fake_labels else '') + '/' + ('legend' if display_legends else 'no_legend') + '/'
plot_directory += ('calibrated' if use_calibration else 'uncalibrated') + '/n_demos_' + str(n_demos) + '/' + confidence_type + '/'

In [9]:
# Define lambdas and epsilons
# NOTE: If running with loss_01_conversion=max_0, cannot have epsilon < 0!
stepsize = 0.01
eps_grid = np.arange(0.0, max_eps + stepsize, stepsize)
lambdas = np.arange(0.0, 1.0 + stepsize, stepsize)[::-1]

In [10]:
# Writing out lambdas for use in Slurm script for CALM experiments
with open("lams.txt", "w") as f:
    for l in lambdas:
        f.write(f"{l:.3f}\n")

In [11]:
# Define loss and risk-control parameters
relative_labels = 'zeroshot_full_model' # zeroshot_full_model or full_model; the predictions over which to compute a relative loss
ground_truth_type = 'true_label' # the ground-truth for computing loss; true_label or zeroshot_full_model
rcp_type = 'ltt'
delta=0.1

In [12]:
if debug_mode:
    # Select just a subset of the datasets and models
    datasets = ['financial_phrasebank']
    n_trials=2
else:
    # Prevent displaying figures
    matplotlib.use('Agg')
    n_trials=50

# Pre-Compute Risk Control Results
This makes plotting much more efficient. 

In [14]:
# Pre-compute and save the risk control results matrices
os.makedirs(precomputed_risk_path, exist_ok=True)
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        scaling_correct = precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/correct/rcp_lams.npy'
        scaling_incorrect = precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/incorrect/rcp_lams.npy'
        if os.path.isfile(scaling_correct) and os.path.isfile(scaling_incorrect):
            continue
        label_order = get_label_order(dataset, tokenizer)
        if fake_labels:
            label_order = [fake_label_map[x] for x in label_order]

        base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
        # First check that there exists all types of experiments
        if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                and os.path.exists(base_dir + 'zeroshot.json')):
            # We have all the data
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

            for gt, rel, label in zip([c_gt, i_gt], [c_rel, i_rel], ['correct', 'incorrect']):
                conf = get_all_confidences(data[label], n_early_exit, label_order, confidence_type, int(n_early_exit/2))
                acc = get_all_accuracies(data[label], gt, n_early_exit, int(n_early_exit/2))
                n_cal = int(len(data[label]['0'])/2)
                # Run the max-0 method
                losses, test_risk, eff_gains, rcp_lams = apply_risk_control(conf, acc, gt, rel, lambdas, eps_grid,
                                                                            delta, n_cal, n_trials, 'max_0')
                # Save results
                max0_path = precomputed_risk_path + 'max0/' + dataset + '/' + model_name + '/' + label + '/'
                if not os.path.exists(max0_path):
                    os.makedirs(max0_path)
                np.save(max0_path + 'losses.npy', losses)
                np.save(max0_path + 'test_risk.npy', test_risk)
                np.save(max0_path + 'eff_gains.npy', eff_gains)
                np.save(max0_path + 'rcp_lams.npy', rcp_lams)
                # Run the scaling method
                losses, test_risk, eff_gains, rcp_lams = apply_risk_control(conf, acc, gt, rel, lambdas, eps_grid,
                                                                            delta, n_cal, n_trials, 'scaling')
                # Save results
                scaling_path = precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/' + label + '/'
                if not os.path.exists(scaling_path):
                    os.makedirs(scaling_path)
                np.save(scaling_path + 'losses.npy', losses)
                np.save(scaling_path + 'test_risk.npy', test_risk)
                np.save(scaling_path + 'eff_gains.npy', eff_gains)
                np.save(scaling_path + 'rcp_lams.npy', rcp_lams)
        else:
            print('Missing data: ', dataset, model_name)
    print('Finished', dataset)


Finished sst2
Finished trec
Missing data:  financial_phrasebank Qwen/Qwen3-8B
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


In [15]:
def filter(dat, idx):
    return [dat[i] for i in idx if 0 <= i < len(dat)]

In [16]:
# Pre-compute and save the risk control results matrices for combined correct + incorrect demos
os.makedirs(precomputed_mixed_path, exist_ok=True)
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'
        if os.path.isfile(scaling_path + 'rcp_lams.npy'):
            continue
        label_order = get_label_order(dataset, tokenizer)
        if fake_labels:
            label_order = [fake_label_map[x] for x in label_order]

        base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
        # First check that there exists all types of experiments
        if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                and os.path.exists(base_dir + 'zeroshot.json')):
            # We have all the data
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

            # Sample proportions of the data as needed
            c_idx = random.sample(range(len(c_gt)), int(c_prop/100 * len(c_gt)))
            i_idx = random.sample(range(len(i_gt)), int(i_prop/100 * len(i_gt)))
            c_gt, i_gt, c_rel, i_rel = filter(c_gt, c_idx), filter(i_gt, i_idx), filter(c_rel, c_idx), filter(i_rel, i_idx)
            cd, id = {}, {}
            for col in data['correct']:
                cd[col] = filter(data['correct'][col], c_idx)
                id[col] = filter(data['incorrect'][col], i_idx)
            data['correct'], data['incorrect'] = cd, id

            # Combine correct and incorrect
            gt, rel = c_gt + i_gt, c_rel + i_rel
            combined_data = {}
            for col in data['correct']:
                combined_data[col] = data['correct'][col] + data['incorrect'][col]

            conf = get_all_confidences(combined_data, n_early_exit, label_order, confidence_type, int(n_early_exit/2))
            acc = get_all_accuracies(combined_data, gt, n_early_exit, int(n_early_exit/2))
            n_cal = int(len(combined_data['0'])/2)

            # Run the max-0 method
            losses, test_risk, eff_gains, rcp_lams = apply_risk_control(conf, acc, gt, rel, lambdas, eps_grid,
                                                                        delta, n_cal, n_trials, 'max_0')
            # Save confidence + accuracy matrices and labels
            conf_acc_path = precomputed_mixed_path + dataset + '/' + model_name + '/'
            if not os.path.exists(conf_acc_path):
                os.makedirs(conf_acc_path)
            np.save(conf_acc_path + 'conf.npy', conf)
            np.save(conf_acc_path + 'acc.npy', acc)
            with open(conf_acc_path + 'true_labels.txt', "w") as file:
                for item in gt:
                    file.write(f"{item}\n")

            with open(conf_acc_path + 'relative_labels.txt', "w") as file:
                for item in rel:
                    file.write(f"{item}\n")

            # Save results
            max0_path = precomputed_mixed_path + 'max0/' + dataset + '/' + model_name + '/'
            if not os.path.exists(max0_path):
                os.makedirs(max0_path)
            np.save(max0_path + 'losses.npy', losses)
            np.save(max0_path + 'test_risk.npy', test_risk)
            np.save(max0_path + 'eff_gains.npy', eff_gains)
            np.save(max0_path + 'rcp_lams.npy', rcp_lams)
            # Run the scaling method
            losses, test_risk, eff_gains, rcp_lams = apply_risk_control(conf, acc, gt, rel, lambdas, eps_grid,
                                                                        delta, n_cal, n_trials, 'scaling')
            if not os.path.exists(scaling_path):
                os.makedirs(scaling_path)
            np.save(scaling_path + 'losses.npy', losses)
            np.save(scaling_path + 'test_risk.npy', test_risk)
            np.save(scaling_path + 'eff_gains.npy', eff_gains)
            np.save(scaling_path + 'rcp_lams.npy', rcp_lams)

        else:
            print('Missing data: ', dataset, model_name)
    print('Finished', dataset)


Finished sst2
Finished trec
Missing data:  financial_phrasebank Qwen/Qwen3-8B
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Plotting Helper Functions

In [18]:
def compute_rc_fixed_lambda(eps_grid, rcp_type, rcp_lams, losses, exits):
    test_risk_e, test_risk_err, eff_gains_e, eff_gains_err = [], [], [], []
    for e, eps in enumerate(eps_grid):
        lam_id = rcp_lams[e]
        if lam_id is None:
            test_risk_e.append(default_loss_uncontrolled_risk)
            eff_gains_e.append(default_eff_gain_uncontrolled_risk)
            test_risk_err.append(0)
            eff_gains_err.append(0)
        else:
            test_risk_e.append(losses[lam_id].mean())
            eff_gains_e.append(exits[lam_id].mean())
            test_risk_err.append(losses[lam_id].std(axis=0) / np.sqrt(losses[lam_id].shape[0]))
            eff_gains_err.append(exits[lam_id].std(axis=0) / np.sqrt(exits[lam_id].shape[0]))

    return np.array(test_risk_e), np.array(eff_gains_e), np.array(test_risk_err), np.array(eff_gains_err)

In [19]:
def plot_risk_control(ax, eps_grid, losses, test_risk, label, color, linestyle, draw_diagonal=True):
    risk_mean, risk_err = test_risk.mean(axis=0), test_risk.std(axis=0) / np.sqrt(test_risk.shape[0])
    # Limit to the available epsilons
    risk_mean, risk_err = risk_mean[:len(eps_grid)], risk_err[:len(eps_grid)]
    ax.fill_between(eps_grid, risk_mean - risk_err, risk_mean + risk_err, alpha=0.2, color=color, zorder=1)
    ax.plot(eps_grid, risk_mean, label=label, color=color, linestyle=linestyle, zorder=3)
    if draw_diagonal:
        ax.plot([min(eps_grid), max(eps_grid)], [min(eps_grid), max(eps_grid)], 'k--', zorder=0)
    ax.set_ylabel('Risk')
    ax.set_xlabel('epsilon')

In [20]:
def plot_separate_risk(ax, eps_grid, c_mean, c_err, i_mean, i_err, color):
    ax.plot(eps_grid, c_mean, label='correct', color=color, linestyle='solid')
    ax.fill_between(eps_grid, c_mean - c_err, c_mean + c_err, alpha=0.2, color=color)
    ax.plot(eps_grid, i_mean, label='incorrect', color=color, linestyle='dotted')
    ax.fill_between(eps_grid, i_mean - i_err, i_mean + i_err, alpha=0.2, color=color)
    ax.set_ylabel('Risk')
    ax.set_xlabel('epsilon')

In [21]:
def plot_efficiency_gains(ax, eps_grid, eff_gains, label, color, linestyle, n_early_exit):
    first_exit = int(n_early_exit / 2)
    exit_mean, exit_err = eff_gains.mean(axis=0), eff_gains.std(axis=0) / np.sqrt(eff_gains.shape[0])
    exit_mean, exit_err = exit_mean[:len(eps_grid)], exit_err[:len(eps_grid)]
    pct_mean = 100.0 * (first_exit + exit_mean) / n_early_exit
    pct_err = 100.0 * exit_err / n_early_exit
    ax.plot(eps_grid, pct_mean, label=label, color=color, linestyle=linestyle)
    ax.fill_between(eps_grid, pct_mean - pct_err, pct_mean + pct_err, alpha=0.2, color=color)
    ax.set_ylabel('Average % of Layers Used')
    ax.set_xlabel('epsilon')

In [22]:
def plot_lambda_vs_risk(ax, lambdas, losses, label, color, linestyle):
    # Create plot of lambda vs risk
    ax.plot(lambdas, losses.mean(axis=1), label=label, color=color, linestyle=linestyle)
    # Add error bars
    err = np.std(losses, axis=1) / np.sqrt(losses.shape[1])
    ax.fill_between(lambdas, losses.mean(axis=1) - err, losses.mean(axis=1) + err, alpha=0.2, color=color)
    ax.set_xlabel('Lambda')
    ax.set_ylabel('Empirical Risk')

# Save Legends
Save various legends for the plots below, so they do not cover results from the plot.

In [24]:
# Incorrect/correct legend
ic_lines = [Line2D([0], [0], color='black', lw=5, linestyle=sty, alpha=0.5) for sty in ['dashed', 'solid']]
ic_labels = ['incorrect', 'correct']

figlegend = plt.figure()
figlegend.legend(ic_lines, ic_labels, loc='center', fontsize=20, ncol=2)
figlegend.savefig(plot_directory.split("/")[1] + "/legends_only/incorrect_correct_legend.pdf")

In [25]:
# Model legend: only models
model_lines = [Line2D([0], [0], color=model_colors[model], lw=5, linestyle='-',) for model in models]
model_labels = [x.split('/')[1] for x in models]

figlegend = plt.figure()
figlegend.legend(model_lines, model_labels, loc='center', fontsize=20, ncol=len(models))
figlegend.savefig(plot_directory.split("/")[1] + "/legends_only/models_legend.pdf")

In [26]:
# Correct vs incorrect, with and without clipping (appendix figure)
lines1 = [Line2D([0], [0], color=c_color, lw=5, linestyle='solid'), Line2D([0], [0], color=i_color, lw=5, linestyle='solid'), 
              Line2D([0], [0], color=c_color, lw=5, linestyle='dashed'), Line2D([0], [0], color=i_color, lw=5, linestyle='dashed')]
labels1 = ['correct with scaling', 'incorrect with scaling', 'correct with clipping', 'incorrect with clipping']

figlegend = plt.figure()
figlegend.legend(lines1, labels1, loc='center', fontsize=20, ncol=4)
figlegend.savefig(plot_directory.split("/")[1] + "/legends_only/ic_scaled_clipped_legend.pdf")

In [27]:
# Correct vs incorrect, with and without demos (Fig 3)
lines1 = [Line2D([0], [0], color=c_color, lw=5, linestyle='solid'), Line2D([0], [0], color=i_color, lw=5, linestyle='solid'), 
              Line2D([0], [0], color=c_color, lw=5, linestyle='dashed'), Line2D([0], [0], color=i_color, lw=5, linestyle='dashed')]
labels1 = ['correct demos', 'incorrect demos', 'correct full model', 'incorrect full model']

figlegend = plt.figure()
figlegend.legend(lines1, labels1, loc='center', fontsize=20, ncol=4)
figlegend.savefig(plot_directory.split("/")[1] + "/legends_only/c_vs_i_legend.pdf")

In [28]:
# Clipped vs scaled risk
cs_lines = [Line2D([0], [0], color='black', lw=5, linestyle=sty, alpha=0.5) for sty in ['dashed', 'solid']]
cs_labels = ['clipped risk', 'scaled risk']

figlegend = plt.figure()
figlegend.legend(cs_lines, cs_labels, loc='center', fontsize=20, ncol=2)
figlegend.savefig(plot_directory.split("/")[1] + "/legends_only/clipped_scaled_legend.pdf")

# Bar Plots - Losses
This is a small figure in sec. 3.2 that shows why scaling makes sense - by better preserving the underlying distribution. 

In [30]:
if max_eps == 0.5:
    for dataset in datasets:
        all_counts = [0, 0, 0]
        for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
            label_order = get_label_order(dataset, tokenizer)
            if fake_labels:
                label_order = [fake_label_map[x] for x in label_order]
            
            base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
            # First check that there exists all types of experiments
            if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                    and os.path.exists(base_dir + 'zeroshot.json')):
                # We have all the data
                data = {}
                for expt_type in ['correct', 'incorrect', 'zeroshot']:
                    with open(base_dir + expt_type + '.json', 'r') as file:
                        data[expt_type] = json.load(file)
                    
                c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
                c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
    
                # Sample proportions of the data as needed
                c_idx = random.sample(range(len(c_gt)), int(c_prop/100 * len(c_gt))) 
                i_idx = random.sample(range(len(i_gt)), int(i_prop/100 * len(i_gt)))
                c_gt, i_gt, c_rel, i_rel = filter(c_gt, c_idx), filter(i_gt, i_idx), filter(c_rel, c_idx), filter(i_rel, i_idx)
                cd, id = {}, {}
                for col in data['correct']:
                    cd[col] = filter(data['correct'][col], c_idx)
                    id[col] = filter(data['incorrect'][col], i_idx)
                data['correct'], data['incorrect'] = cd, id
    
                # Combine correct and incorrect
                gt, rel = c_gt + i_gt, c_rel + i_rel
                combined_data = {}
                for col in data['correct']:
                    combined_data[col] = data['correct'][col] + data['incorrect'][col]
    
                conf = get_all_confidences(combined_data, n_early_exit, label_order, confidence_type, int(n_early_exit/2))
                acc = get_all_accuracies(combined_data, gt, n_early_exit, int(n_early_exit/2))
                n_cal = int(len(combined_data['0'])/2)
                
                losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, rel, gt, True)
                unique_values, counts = np.unique(losses, return_counts=True)
                # Create mapping from value to index in all_counts; note that unique_values may not always contain all of -1, 0, 1
                mapping = {-1: 0, 0: 1, 1: 2}
                for val, c in zip(unique_values, counts):
                    all_counts[mapping[val]] += c
    
        all_counts = all_counts / sum(all_counts) * 100
        
        # Create the bar chart
        fig, ax = plt.subplots(1,1,figsize=(5,5))
        ax.bar(unique_values, all_counts, width=0.6, color='skyblue', edgecolor='black')
        ax.set_xticks([-1, 0, 1])
        
        # Add labels and title
        ax.set_xlabel('Loss')
        ax.set_ylabel('% of Data')
        ax.set_title('Context-Aware Losses')
    
        if debug_mode:
            # Display the image
            plt.show()
        else:
            # Save out the image
            path = plot_directory + mixed_filename + '/losses_barplot/' 
            if not os.path.exists(path):
                os.makedirs(path)
            plt.savefig(path + dataset + '_scaled.pdf')
    
        # Create the bar chart for clipped losses
        fig, ax = plt.subplots(1,1,figsize=(5,5))
        print(all_counts)
        ax.bar([0,1], [all_counts[0] + all_counts[1], all_counts[2]], width=0.6, color='skyblue', edgecolor='black')
        ax.set_xticks([-1, 0, 1])
        
        # Add labels and title
        ax.set_xlabel('Loss')
        ax.set_ylabel('% of Data')
        ax.set_title('Clipped Context-Aware Losses')
    
        if debug_mode:
            # Display the image
            plt.show()
        else:
            # Save out the image
            path = plot_directory + mixed_filename + '/losses_barplot/' 
            if not os.path.exists(path):
                os.makedirs(path)
            plt.savefig(path + dataset + '_clipped.pdf')
    
        print("Finished", dataset)

[10.59987095 53.58527372 35.81485533]
Finished sst2
[17.73187286 69.79266033 12.47546682]
Finished trec
[20.25022039 53.20051873 26.54926089]
Finished financial_phrasebank
[25.49393718 44.74203623 29.76402659]
Finished tweeteval_hate
[22.53165722 52.4576718  25.01067097]
Finished tweeteval_feminist
[21.28484738 54.14062144 24.57453119]
Finished tweeteval_atheism
[14.56388496 43.99182775 41.44428729]
Finished unnatural
[10.74465868 71.24484479 18.01049654]
Finished ag_news


/var/folders/1g/wx_xh_xx44bdx2g8gcx64pbr0000gn/T/ipykernel_22570/3365806657.py:72: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(1,1,figsize=(5,5))


## Incorrect/Correct Sub-group Plots - Risk Control
Paper: fig 6, 13, 14

In [32]:
# Split plots - scaling or max0
for subgroup_plot_type in ['scaling', 'max0']:
    print(subgroup_plot_type, '------------------------')
    for dataset in datasets:
        fig, ax = plt.subplots(1,1,figsize=(5,5))
        for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
            # First check that all experiment results are precomputed
            if os.path.exists(precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'):
                label_order = get_label_order(dataset, tokenizer)
                if fake_labels:
                    label_order = [fake_label_map[x] for x in label_order]
                
                # load the data
                base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
                data = {}
                for expt_type in ['correct', 'incorrect', 'zeroshot']:
                    with open(base_dir + expt_type + '.json', 'r') as file:
                        data[expt_type] = json.load(file)
    
                c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
                c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
    
                # Combine correct and incorrect
                gt, rel = c_gt + i_gt, c_rel + i_rel
                combined_data = {}
                for col in data['correct']:
                    combined_data[col] = data['correct'][col] + data['incorrect'][col]
                
                # get the pre-computed lambda-hat's
                scaling_path = precomputed_mixed_path + subgroup_plot_type + '/' + dataset + '/' + model_name + '/'
                rcp_lams = np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
    
                # compute + plot epsilon vs risk for correct demos
                conf = get_all_confidences(data['correct'], n_early_exit, label_order, confidence_type, int(n_early_exit/2))
                acc = get_all_accuracies(data['correct'], c_gt, n_early_exit, int(n_early_exit/2))
                losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, c_rel, c_gt)
                c_mean, _, c_err, _ = compute_rc_fixed_lambda(eps_grid, rcp_type, rcp_lams, losses, exits)
    
                # compute + plot epsilon vs risk for incorrect demos
                conf = get_all_confidences(data['incorrect'], n_early_exit, label_order, confidence_type, int(n_early_exit/2))
                acc = get_all_accuracies(data['incorrect'], c_gt, n_early_exit, int(n_early_exit/2))
                losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, c_rel, c_gt)
                i_mean, _, i_err, _ = compute_rc_fixed_lambda(eps_grid, rcp_type, rcp_lams, losses, exits)
    
                # plot correct/incorrect separately
                plot_separate_risk(ax, eps_grid, c_mean, c_err, i_mean, i_err, model_colors[model_name])
            else:
                print('Missing data:', model_name, dataset)

        if display_legends:
            lines2 = [Line2D([0], [0], color='black', lw=5, linestyle=sty, alpha=0.5) for sty in ['dashed', 'solid']]
            labels2 = ['incorrect', 'correct']
            
            lines1 = [Line2D([0], [0], color=model_colors[model], lw=5, linestyle='-',) for model in models]
            lines1 += lines2
            labels1 = [x.split('/')[1] for x in models] + labels2
            
            # legend1 = plt.legend(lines1, labels1, loc='upper left',)
            # Save the legend as its own separate thing
            figlegend = plt.figure()
            figlegend.legend(lines1, labels1, loc='center', fontsize=20)
            
        ax.plot([min(eps_grid), max(eps_grid)], [min(eps_grid), max(eps_grid)], 'k--')
        plt.tight_layout()
    
        if debug_mode:
            # Display the image
            plt.show()
        else:
            # Save out the image
            path = plot_directory + mixed_filename + '/rc_split_' + subgroup_plot_type + '/' 
            if not os.path.exists(path):
                os.makedirs(path)
            plt.savefig(path + dataset + '.pdf')
            #figlegend.savefig(path + "legend.pdf")
    
            # Close plots to save memory
            matplotlib.pyplot.close()
    
        print('Finished', dataset)

scaling ------------------------
Finished sst2
Finished trec
Missing data: Qwen/Qwen3-8B financial_phrasebank
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news
max0 ------------------------
Finished sst2
Finished trec
Missing data: Qwen/Qwen3-8B financial_phrasebank
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


## Risk Control - Epsilon vs Risk
Paper: fig 5, 9-12

In [34]:
# Risk control plots - with only risk control on combined data
for dataset in datasets:
    fig, ax = plt.subplots(1,1,figsize=(5,5))
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'
        if os.path.exists(scaling_path):
            test_risk, losses = np.load(scaling_path + 'test_risk.npy'), np.load(scaling_path + 'losses.npy')
            plot_risk_control(ax, eps_grid, losses, test_risk, model_name.split('/')[1], model_colors[model_name], 'solid', draw_diagonal=False)
        else:
            print('Missing data:', model_name, dataset)

    ax.plot([min(eps_grid), max(eps_grid)], [min(eps_grid), max(eps_grid)], 'k--', zorder=0)
    if display_legends:
        ax.legend()

    plt.title(dataset_names_plot_titles[dataset])
    plt.tight_layout()

    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + mixed_filename + '/risk_control/'
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + dataset + '.pdf')

    # Close plots to save memory
    matplotlib.pyplot.close()

    print('Finished', dataset)

Finished sst2
Finished trec
Missing data: Qwen/Qwen3-8B financial_phrasebank
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


## Risk Control + Efficiency Gains - Comparison (Clipping vs Our Approach)
Paper: fig 4, 18

In [36]:
# Plot risk control results and exit layer, and with comparison to the other approach
# Also study the average efficiency gains
for dataset in datasets:
    fig, ax = plt.subplots(1,2,figsize=(10,5))
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        if os.path.exists(precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'):
            label_order = get_label_order(dataset, tokenizer)
            if fake_labels:
                label_order = [fake_label_map[x] for x in label_order]
            
            base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
            # We have all the data
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

            # Combine correct and incorrect
            gt, rel = c_gt + i_gt, c_rel + i_rel
            combined_data = {}
            for col in data['correct']:
                combined_data[col] = data['correct'][col] + data['incorrect'][col]
            
            # max-0
            max0_path = precomputed_mixed_path + 'max0/' + dataset + '/' + model_name + '/'
            #eff_gains, rcp_lams = np.load(max0_path + 'eff_gains.npy'), np.load(max0_path + 'rcp_lams.npy', allow_pickle=True)
            losses, test_risk = np.load(max0_path + 'losses.npy'), np.load(max0_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(max0_path + 'eff_gains.npy'), np.load(max0_path + 'rcp_lams.npy', allow_pickle=True)
            # using the rcp_lams compute risk
            # conf = get_all_confidences(combined_data, n_early_exit, label_order, confidence_type, int(n_early_exit/2))
            # acc = get_all_accuracies(combined_data, gt, n_early_exit, int(n_early_exit/2))
            # losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, rel, gt)
            # mean, _, err, _ = compute_rc_fixed_lambda(eps_grid, rcp_type, rcp_lams, losses, exits)
            # mean, err = mean[:len(eps_grid)], err[:len(eps_grid)]
            # ax[0].plot(eps_grid, mean, label='clipped risk', color=model_colors[model_name], linestyle='dashed')
            # ax[0].fill_between(eps_grid, mean - err, mean + err, alpha=0.2, color=model_colors[model_name])
            plot_risk_control(ax[0], eps_grid, losses, test_risk, 'clipped risk', model_colors[model_name], 'dashed')
            plot_efficiency_gains(ax[1], eps_grid, eff_gains, 'clipped risk', model_colors[model_name], 'dashed', n_early_exit)
            # using the rcp_lams compute
            scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/' 
            losses, test_risk = np.load(scaling_path + 'losses.npy'), np.load(scaling_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(scaling_path + 'eff_gains.npy'), np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
            plot_risk_control(ax[0], eps_grid, losses, test_risk, 'risk transformation', model_colors[model_name], 'solid')
            plot_efficiency_gains(ax[1], eps_grid, eff_gains, 'risk transformation', model_colors[model_name], 'solid', n_early_exit)
        else:
            print('Missing data: ', dataset, model_name)

    if display_legends:
        lines2 = [Line2D([0], [0], color='black', lw=5, linestyle=sty, alpha=0.5) for sty in ['dashed', 'solid']]
        labels2 = ['clipped risk', 'risk transformation']
        
        lines1 = [Line2D([0], [0], color=model_colors[model], lw=5, linestyle='-',) for model in models]
        labels1 = [x.split('/')[1] for x in models]
        
        legend1 = ax[0].legend(lines1, labels1, loc='upper left',)
        legend2 = ax[1].legend(lines2, labels2, loc='upper right',)
    
    plt.tight_layout()
    
    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + mixed_filename + '/risk_control_with_comparison/'
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + dataset + '.pdf')

    # Close plots to save memory
    matplotlib.pyplot.close()

    print('Finished', dataset)

Finished sst2
Finished trec
Missing data:  financial_phrasebank Qwen/Qwen3-8B
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Risk Control Only - Comparison (Scaling vs Clipping)

In [38]:
# Plot risk control results with comparison to the other approach
for dataset in datasets:
    fig, ax = plt.subplots(1,1,figsize=(5,5))
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        if os.path.exists(precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'):
            label_order = get_label_order(dataset, tokenizer)
            if fake_labels:
                label_order = [fake_label_map[x] for x in label_order]
            
            base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
            # We have all the data
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

            # Combine correct and incorrect
            gt, rel = c_gt + i_gt, c_rel + i_rel
            combined_data = {}
            for col in data['correct']:
                combined_data[col] = data['correct'][col] + data['incorrect'][col]
            
            # max-0
            max0_path = precomputed_mixed_path + 'max0/' + dataset + '/' + model_name + '/'
            #eff_gains, rcp_lams = np.load(max0_path + 'eff_gains.npy'), np.load(max0_path + 'rcp_lams.npy', allow_pickle=True)
            losses, test_risk = np.load(max0_path + 'losses.npy'), np.load(max0_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(max0_path + 'eff_gains.npy'), np.load(max0_path + 'rcp_lams.npy', allow_pickle=True)
            plot_risk_control(ax, eps_grid, losses, test_risk, 'clipped risk', model_colors[model_name], 'dashed')
            # using the rcp_lams compute
            scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/' 
            losses, test_risk = np.load(scaling_path + 'losses.npy'), np.load(scaling_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(scaling_path + 'eff_gains.npy'), np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
            plot_risk_control(ax, eps_grid, losses, test_risk, 'risk transformation', model_colors[model_name], 'solid')
        else:
            print('Missing data: ', dataset, model_name)

    if display_legends:
        lines2 = [Line2D([0], [0], color='black', lw=5, linestyle=sty, alpha=0.5) for sty in ['dashed', 'solid']]
        labels2 = ['clipped risk', 'risk transformation']
        
        lines1 = [Line2D([0], [0], color=model_colors[model], lw=5, linestyle='-',) for model in models]
        labels1 = [x.split('/')[1] for x in models]
        
        legend1 = ax[0].legend(lines1, labels1, loc='upper left',)
        legend2 = ax[1].legend(lines2, labels2, loc='upper right',)
    
    plt.tight_layout()
    
    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + mixed_filename + '/risk_control_only_comparison/'
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + dataset + '.pdf')

        # Close plots to save memory
        matplotlib.pyplot.close()

    print('Finished', dataset)

Finished sst2
Finished trec
Missing data:  financial_phrasebank Qwen/Qwen3-8B
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


## Efficiency Gains - Comparison (Clipping vs Our Approach)
Paper: fig 16

In [40]:
# Plot JUST efficiency gains
for dataset in datasets:
    fig, ax = plt.subplots(1,1,figsize=(5,5))
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        if os.path.exists(precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'):
            scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'
            rcp_lams = np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
            
            # max-0
            max0_path = precomputed_mixed_path + 'max0/' + dataset + '/' + model_name + '/'
            losses, test_risk = np.load(max0_path + 'losses.npy'), np.load(max0_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(max0_path + 'eff_gains.npy'), np.load(max0_path + 'rcp_lams.npy', allow_pickle=True)
            plot_efficiency_gains(ax, eps_grid, eff_gains, 'clipped risk', model_colors[model_name], 'dashed', n_early_exit)
            # Scaling
            scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/' 
            losses, test_risk = np.load(scaling_path + 'losses.npy'), np.load(scaling_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(scaling_path + 'eff_gains.npy'), np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
            plot_efficiency_gains(ax, eps_grid, eff_gains, 'risk transformation', model_colors[model_name], 'solid', n_early_exit)
        else:
            print('Missing data: ', dataset, model_name)

    if display_legends:
        lines2 = [Line2D([0], [0], color='black', lw=5, linestyle=sty, alpha=0.5) for sty in ['dashed', 'solid']]
        labels2 = ['clipped risk', 'risk transformation']
        
        lines1 = [Line2D([0], [0], color=model_colors[model], lw=5, linestyle='-',) for model in models] + lines2
        labels1 = [x.split('/')[1] for x in models] + labels2
        
        legend1 = ax.legend(lines1, labels1, loc='upper left',)

    plt.title(dataset_names_plot_titles[dataset])
    plt.tight_layout()
    
    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + mixed_filename + '/eff_gains/'
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + dataset + '.pdf')

        # Close plots to save memory
        matplotlib.pyplot.close()

    print('Finished', dataset)

Finished sst2
Finished trec
Missing data:  financial_phrasebank Qwen/Qwen3-8B
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Highlighted Lambda vs Accuracy
Paper: fig 3

In [42]:
# Plot lambda vs relative accuracy
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        fig, ax = plt.subplots(1,1,figsize=(5,5))
        label_order = get_label_order(dataset, tokenizer)
        if fake_labels:
            label_order = [fake_label_map[x] for x in label_order]
        
        base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
        # First check that there exists all types of experiments
        if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                and os.path.exists(base_dir + 'zeroshot.json')):
            # We have all the data
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

            # Plot lambda vs correct accuracy
            conf = get_all_confidences(data['correct'], n_early_exit, label_order, confidence_type, int(n_early_exit/2))
            acc = get_all_accuracies(data['correct'], c_gt, n_early_exit, int(n_early_exit/2))
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, None, c_gt, default_to_zero_shot=False)
            c_acc = 1-losses.mean(axis=1)
            ax.plot(lambdas, c_acc, label='correct demos', color=c_color)

            # Plot lambda vs incorrect accuracy
            conf = get_all_confidences(data['incorrect'], n_early_exit, label_order, confidence_type, int(n_early_exit/2))
            acc = get_all_accuracies(data['incorrect'], c_gt, n_early_exit, int(n_early_exit/2))
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, None, i_gt, default_to_zero_shot=False)
            i_acc = 1-losses.mean(axis=1)
            ax.plot(lambdas, i_acc, label='incorrect demos', color=i_color)

            # Plot full-model performance as horizontal lines
            correct_acc = [1 if p == t else 0 for p,t in zip(data['correct'][str(n_early_exit-1)], c_gt)]
            incorrect_acc = [1 if p == t else 0 for p,t in zip(data['incorrect'][str(n_early_exit-1)], i_gt)]
            zeroshot_acc = [1 if p == t else 0 for p,t in zip(data['zeroshot'][str(n_early_exit-1)], z_gt)]
            correct_rel = [c-z for c,z in zip(correct_acc, zeroshot_acc)]
            incorrect_rel = [i-z for i,z in zip(incorrect_acc, zeroshot_acc)]
            avg_correct_rel, avg_incorrect_rel = c_acc[0], i_acc[0]
            # Plot the full-model accuracies
            ax.axhline(y=avg_correct_rel, color=c_color, linestyle='--', linewidth=2, label='correct full model')
            ax.axhline(y=avg_incorrect_rel, color=i_color, linestyle='--', linewidth=2, label='incorrect full model')
            valid_lams = []
            for lam_idx in range(len(lambdas)):
                if (avg_correct_rel - c_acc[len(lambdas)-lam_idx-1]) < 0.05 and i_acc[len(lambdas)-lam_idx-1] > avg_incorrect_rel:
                    valid_lams.append(len(lambdas)-lam_idx-1)

            intervals_idx = [(g[0][1], g[-1][1]) for _, group in groupby(enumerate(valid_lams), lambda x: x[1] - x[0]) for g in [list(group)]]
            for start,end in intervals_idx:
                end = end+1 if start == end and end < len(lambdas)-1 else end
                plt.axvspan(lambdas[start], lambdas[end], facecolor='yellow', alpha=0.3)
            
            ax.set_xlabel('Lambda')
            ax.set_ylabel('Accuracy')
            if display_legends:
                ax.legend(loc='upper left')
            
            plt.tight_layout()
            
            if debug_mode:
                # Display the image
                plt.show()
            else:
                # Save out the image
                path = plot_directory + 'lambda_vs_accuracy/'
                if not os.path.exists(path):
                    os.makedirs(path)
                plt.savefig(path + dataset + '_' + model_name.split('/')[1] + '.pdf')
    
                # Close plots to save memory
                matplotlib.pyplot.close()
        else:
            print('Missing data: ', dataset, model_name)
    
    print('Finished', dataset)

Finished sst2
Finished trec
Missing data:  financial_phrasebank Qwen/Qwen3-8B
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Rebuttals - Highlighted Lambda vs Accuracy Aggregated Across Datasets

In [44]:
# Plot lambda vs relative accuracy
for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
    data = {}
    longest_label_order = []
    fig, ax = plt.subplots(1,1,figsize=(5,5))
    for dataset in datasets:
        label_order = get_label_order(dataset, tokenizer)
        if fake_labels:
            label_order = [fake_label_map[x] for x in label_order]
        if len(label_order) > len(longest_label_order):
            longest_label_order = label_order
        
        base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
        # First check that there exists all types of experiments
        if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                and os.path.exists(base_dir + 'zeroshot.json')):
            # We have all the data
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    single_dataset = json.load(file)
                if expt_type not in data:
                    data[expt_type] = single_dataset
                else:
                    for key in data[expt_type]:
                        data[expt_type][key] = data[expt_type][key] + single_dataset[key]

    c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
    c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

    # Plot lambda vs correct accuracy
    conf = get_all_confidences(data['correct'], n_early_exit, longest_label_order, confidence_type, int(n_early_exit/2))
    acc = get_all_accuracies(data['correct'], c_gt, n_early_exit, int(n_early_exit/2))
    losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, None, c_gt, default_to_zero_shot=False)
    c_acc = 1-losses.mean(axis=1)
    ax.plot(lambdas, c_acc, label='correct demos', color=c_color)

    # Plot lambda vs incorrect accuracy
    conf = get_all_confidences(data['incorrect'], n_early_exit, longest_label_order, confidence_type, int(n_early_exit/2))
    acc = get_all_accuracies(data['incorrect'], c_gt, n_early_exit, int(n_early_exit/2))
    losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, None, i_gt, default_to_zero_shot=False)
    i_acc = 1-losses.mean(axis=1)
    ax.plot(lambdas, i_acc, label='incorrect demos', color=i_color)

    # Plot full-model performance as horizontal lines
    correct_acc = [1 if p == t else 0 for p,t in zip(data['correct'][str(n_early_exit-1)], c_gt)]
    incorrect_acc = [1 if p == t else 0 for p,t in zip(data['incorrect'][str(n_early_exit-1)], i_gt)]
    zeroshot_acc = [1 if p == t else 0 for p,t in zip(data['zeroshot'][str(n_early_exit-1)], z_gt)]
    correct_rel = [c-z for c,z in zip(correct_acc, zeroshot_acc)]
    incorrect_rel = [i-z for i,z in zip(incorrect_acc, zeroshot_acc)]
    avg_correct_rel, avg_incorrect_rel = c_acc[0], i_acc[0]
    # Plot the full-model accuracies
    ax.axhline(y=avg_correct_rel, color=c_color, linestyle='--', linewidth=2, label='correct full model')
    ax.axhline(y=avg_incorrect_rel, color=i_color, linestyle='--', linewidth=2, label='incorrect full model')
    valid_lams = []
    for lam_idx in range(len(lambdas)):
        if (avg_correct_rel - c_acc[len(lambdas)-lam_idx-1]) < 0.05 and i_acc[len(lambdas)-lam_idx-1] > avg_incorrect_rel:
            valid_lams.append(len(lambdas)-lam_idx-1)

    intervals_idx = [(g[0][1], g[-1][1]) for _, group in groupby(enumerate(valid_lams), lambda x: x[1] - x[0]) for g in [list(group)]]
    for start,end in intervals_idx:
        end = end+1 if start == end and end < len(lambdas)-1 else end
        plt.axvspan(lambdas[start], lambdas[end], facecolor='yellow', alpha=0.3)
    
    ax.set_xlabel('Lambda')
    ax.set_ylabel('Accuracy')
    if display_legends:
        ax.legend(loc='upper left')
    ax.set_title(model_name.split('/')[1] + ' Aggregated over All Datasets')
    
    plt.tight_layout()
    
    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + 'lambda_vs_accuracy_aggregated/'
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + model_name.split('/')[1] + '.png')

        # Close plots to save memory
        matplotlib.pyplot.close()
        
    print('Finished', model_name)

Finished facebook/layerskip-llama3-8B
Finished facebook/layerskip-llama2-7B
Finished meta-llama/Meta-Llama-3-8B
Finished meta-llama/Llama-2-7B-hf
Finished allenai/Olmo-3-1125-32B
Finished allenai/Olmo-3-1025-7B
Finished Qwen/Qwen3-8B


# Highlighted Lambda vs Accuracy with Error Bars
Paper: fig 8

In [46]:
# Plot lambda vs relative accuracy
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        fig, ax = plt.subplots(1,1,figsize=(5,5))
        label_order = get_label_order(dataset, tokenizer)
        if fake_labels:
            label_order = [fake_label_map[x] for x in label_order]
        
        base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
        # First check that there exists all types of experiments
        if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                and os.path.exists(base_dir + 'zeroshot.json')):
            # We have all the data
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

            # Plot lambda vs correct accuracy
            conf = get_all_confidences(data['correct'], n_early_exit, label_order, confidence_type, int(n_early_exit/2))
            acc = get_all_accuracies(data['correct'], c_gt, n_early_exit, int(n_early_exit/2))
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, None, c_gt, False)
            c_acc, c_err = 1-losses.mean(axis=1), losses.std(axis=1)/np.sqrt(losses.shape[1])
            ax.plot(lambdas, c_acc, label='correct demos', color=c_color)
            ax.fill_between(lambdas, c_acc-c_err, c_acc+c_err, color=c_color, alpha=0.3)

            # Plot lambda vs incorrect accuracy
            conf = get_all_confidences(data['incorrect'], n_early_exit, label_order, confidence_type, int(n_early_exit/2))
            acc = get_all_accuracies(data['incorrect'], c_gt, n_early_exit, int(n_early_exit/2))
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, None, i_gt, False)
            i_acc, i_err = 1-losses.mean(axis=1), losses.std(axis=1)/np.sqrt(losses.shape[1])
            ax.plot(lambdas, i_acc, label='incorrect demos', color=i_color)
            ax.fill_between(lambdas, i_acc-i_err, i_acc+i_err, color=i_color, alpha=0.3)

            # Plot full-model performance as horizontal lines
            correct_acc = [1 if p == t else 0 for p,t in zip(data['correct'][str(n_early_exit-1)], c_gt)]
            incorrect_acc = [1 if p == t else 0 for p,t in zip(data['incorrect'][str(n_early_exit-1)], i_gt)]
            zeroshot_acc = [1 if p == t else 0 for p,t in zip(data['zeroshot'][str(n_early_exit-1)], z_gt)]
            correct_rel = [c-z for c,z in zip(correct_acc, zeroshot_acc)]
            incorrect_rel = [i-z for i,z in zip(incorrect_acc, zeroshot_acc)]
            avg_correct_rel, avg_incorrect_rel = c_acc[0], i_acc[0]
            # Plot the full-model accuracies
            ax.axhline(y=avg_correct_rel, color=c_color, linestyle='--', linewidth=2, label='correct full model')
            ax.axhline(y=avg_incorrect_rel, color=i_color, linestyle='--', linewidth=2, label='incorrect full model')
            
            valid_lams = []
            for lam_idx in range(len(lambdas)):
                idx = len(lambdas)-lam_idx-1
                if (avg_correct_rel - c_acc[idx]) - c_err[idx] < 0.05 and i_acc[idx] - i_err[idx] > avg_incorrect_rel:
                    valid_lams.append(len(lambdas)-lam_idx-1)

            intervals_idx = [(g[0][1], g[-1][1]) for _, group in groupby(enumerate(valid_lams), lambda x: x[1] - x[0]) for g in [list(group)]]
            for start,end in intervals_idx:
                end = end+1 if start == end and end < len(lambdas)-1 else end
                plt.axvspan(lambdas[start], lambdas[end], facecolor='yellow', alpha=0.3)
            
            ax.set_xlabel('Lambda')
            ax.set_ylabel('Accuracy')
            if display_legends:
                ax.legend(loc='upper left')
                
            plt.tight_layout()
            
            if debug_mode:
                # Display the image
                plt.show()
            else:
                # Save out the image
                path = plot_directory + 'lambda_vs_accuracy/with_error_bars/' 
                if not os.path.exists(path):
                    os.makedirs(path)
                plt.savefig(path + dataset + '_' + model_name.split('/')[1] + '.pdf')
    
                # Close plots to save memory
                matplotlib.pyplot.close()
        else:
            print('Missing data: ', dataset, model_name)
    
    print('Finished', dataset)

Finished sst2
Finished trec
Missing data:  financial_phrasebank Qwen/Qwen3-8B
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Non-Monotonicity of Risk
Paper: fig 7

In [48]:
# Show non-monotonicity of risk
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        fig, ax = plt.subplots(1,1,figsize=(5,5))
        label_order = get_label_order(dataset, tokenizer)
        if fake_labels:
            label_order = [fake_label_map[x] for x in label_order]
        
        base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
        # First check that there exists all types of experiments
        if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                and os.path.exists(base_dir + 'zeroshot.json')):
            # We have all the data
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

            # Combine correct and incorrect
            gt, rel = c_gt + i_gt, c_rel + i_rel
            combined_data = {}
            for col in data['correct']:
                combined_data[col] = data['correct'][col] + data['incorrect'][col]

            # Plot lambda vs overall accuracy for this model
            conf = get_all_confidences(combined_data, n_early_exit, label_order, confidence_type, int(n_early_exit/2))
            acc = get_all_accuracies(combined_data, gt, n_early_exit, int(n_early_exit/2))
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, rel, gt)
            ax.plot(lambdas, 1-losses.mean(axis=1))
            ax.set_xlabel('Lambda')
            ax.set_ylabel('Risk')
            plt.tight_layout()

            if debug_mode:
                # Display the image
                plt.show()
            else:
                # Save out the image
                path = plot_directory + mixed_filename + '/lambda_vs_accuracy/'
                if not os.path.exists(path):
                    os.makedirs(path)
                plt.savefig(path + dataset + '_' + model_name.split('/')[1] + '.pdf')
        
                # Close plots to save memory
                matplotlib.pyplot.close()
        else:
            print('Missing data: ', dataset, model_name)

    print('Finished', dataset)

Finished sst2
Finished trec
Missing data:  financial_phrasebank Qwen/Qwen3-8B
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Separate Correct and Incorrect Demos Results - Risk Control and Efficiency Gains
Not included in paper. 

In [50]:
# Plot risk control results and exit layer
# ONLY using the scaling approach, and not including zero-shot! One plot per dataset. 
for dataset in datasets:
    fig, ax = plt.subplots(1,2,figsize=(10,5))
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        # First check that all experiment results are precomputed
        if os.path.exists(precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/correct'):
            # Run the scaling method ONLY - correct demos
            scaling_path = precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/correct/'
            losses, test_risk = np.load(scaling_path + 'losses.npy'), np.load(scaling_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(scaling_path + 'eff_gains.npy'), np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
            plot_risk_control(ax[0], eps_grid, losses, test_risk, 'correct', model_colors[model_name], 'solid')
            plot_efficiency_gains(ax[1], eps_grid, eff_gains, 'correct', model_colors[model_name], 'solid', n_early_exit)
        
            # Run the scaling method ONLY - incorrect demos
            scaling_path = precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/incorrect/'
            losses, test_risk = np.load(scaling_path + 'losses.npy'), np.load(scaling_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(scaling_path + 'eff_gains.npy'), np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
            plot_risk_control(ax[0], eps_grid, losses, test_risk, 'incorrect', model_colors[model_name], 'dashed')
            plot_efficiency_gains(ax[1], eps_grid, eff_gains, 'incorrect', model_colors[model_name], 'dashed', n_early_exit)
        else:
            print('Missing data: ', dataset, model_name)

    if display_legends:
        lines2 = [Line2D([0], [0], color='black', lw=5, linestyle=sty, alpha=0.5) for sty in ['dashed', 'solid']]
        labels2 = ['incorrect', 'correct']
        
        lines1 = [Line2D([0], [0], color=model_colors[model], lw=5, linestyle='-',) for model in models]
        labels1 = [x.split('/')[1] for x in models]
        
        legend1 = ax[0].legend(lines1, labels1, loc='upper right',)
        legend2 = ax[1].legend(lines2, labels2, loc='upper right',)
    
    plt.tight_layout()
    
    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + 'risk_control/'
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + dataset + '.pdf')

        # Close plots to save memory
        matplotlib.pyplot.close()
    
    print('Finished', dataset)

Finished sst2
Finished trec
Missing data:  financial_phrasebank Qwen/Qwen3-8B
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Compute Accuracy Difference between Clipping and Full Model
Used to compute accuracy delta column in Table 1. 

In [52]:
# Get the accuracy diff between my early exit and the full dataset for each epsilon
avg_acc_diff = [0 for i in range(len(eps_grid))]
total_n = 0
for dataset in datasets:
    print(dataset)
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        # First check that all experiment results are precomputed
        if os.path.exists(precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'):
            label_order = get_label_order(dataset, tokenizer)
            if fake_labels:
                label_order = [fake_label_map[x] for x in label_order]
            
            # load the data
            base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

            # Combine correct and incorrect
            gt, rel = c_gt + i_gt, c_rel + i_rel
            combined_data = {}
            for col in data['correct']:
                combined_data[col] = data['correct'][col] + data['incorrect'][col]

            # get the full-model performance on this model + dataset (relative to zero-shot)
            full_model_risk = [1 if p == t else 0 for p,t in zip(combined_data[str(n_early_exit-1)], gt)]
            zeroshot_risk = [1 if p == t else 0 for p,t in zip(data['zeroshot'][str(n_early_exit-1)], z_gt)]
            zeroshot_risk = sum(zeroshot_risk) / len(zeroshot_risk)
            full_model_risk = sum(full_model_risk) / len(full_model_risk)
            full_model_risk = full_model_risk - zeroshot_risk
            
            # get the pre-computed losses per epsilon
            scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'
            test_risk = np.load(scaling_path + 'test_risk.npy').mean(axis=0)
            avg_acc_diff = [avg_acc_diff[i] + test_risk[i] - full_model_risk for i in range(len(avg_acc_diff))]
            total_n += 1

avg_acc_diff = [i/total_n for i in avg_acc_diff]
# for i in range(len(eps_grid)):
#     print(eps_grid[i], 'Avg acc diff:', avg_acc_diff[i])

sst2
trec
financial_phrasebank
tweeteval_hate
tweeteval_feminist
tweeteval_atheism
unnatural
ag_news


# Compute Efficiency Gains
Used to compute the two efficiency gains columns in Table 1.

In [54]:
# Evaluate efficiency gain difference between scaling vs max0
eps_to_evaluate = 0.05 # must be less than max_eps!
eps_index = np.where(eps_grid == eps_to_evaluate)[0][0]
avg_incorrect_diff, avg_correct_diff, total_n = 0, 0, 0
avg_correct_scaling_gains = 0
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        if os.path.exists(precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/correct/'):
            total_n += 1
            # Correct demos
            eff_gains = np.load(precomputed_risk_path + 'max0/' + dataset + '/' + model_name + '/correct/' + 'eff_gains.npy')
            max0_gains = eff_gains.mean(axis=0)[eps_index]
            eff_gains = np.load(precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/correct/' + 'eff_gains.npy')
            scaling_gains = eff_gains.mean(axis=0)[eps_index]
            losses = np.load(precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/correct/' + 'losses.npy')
            avg_correct_scaling_gains += scaling_gains
            avg_correct_diff += (scaling_gains - max0_gains)
            # Incorrect demos
            eff_gains = np.load(precomputed_risk_path + 'max0/' + dataset + '/' + model_name + '/incorrect/' + 'eff_gains.npy')
            max0_gains = eff_gains.mean(axis=0)[eps_index]
            eff_gains = np.load(precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/incorrect/' + 'eff_gains.npy')
            scaling_gains = eff_gains.mean(axis=0)[eps_index]
            avg_incorrect_diff += (scaling_gains - max0_gains)
        else:
            print('Missing data: ', dataset, model_name)

if total_n > 0:
    avg_correct_diff, avg_incorrect_diff = avg_correct_diff / total_n, avg_incorrect_diff / total_n
    print('Avg diff - correct examples:', avg_correct_diff, 'Percent:', avg_correct_diff/17*100)
    print('Avg diff - incorrect examples:', avg_incorrect_diff, 'Percent:', avg_incorrect_diff/17*100)
    avg_correct_scaling_gains = avg_correct_scaling_gains / total_n
    print('Avg diff vs full model (correct):', avg_correct_scaling_gains, 'Percent:', avg_correct_scaling_gains/17*100)

Missing data:  financial_phrasebank Qwen/Qwen3-8B
Avg diff - correct examples: -8.116002385165817 Percent: -47.74119050097539
Avg diff - incorrect examples: -2.180011059458759 Percent: -12.823594467404465
Avg diff vs full model (correct): 7.983316724351185 Percent: 46.9606866138305


# Efficiency Gains - Correct vs Incorrect Demos & Clipped vs Transformed Risk
Not included in paper.

In [56]:
# Plot risk control results and exit layer, and with comparison to the other approach
# Also study the average efficiency gains
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        if os.path.exists(precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/correct/'):
            fig, ax = plt.subplots(1,1,figsize=(5,5))            
            for label, color in zip(['correct', 'incorrect'], [c_color, i_color]):
                # max-0
                max0_path = precomputed_risk_path + 'max0/' + dataset + '/' + model_name + '/' + label + '/'
                losses, test_risk = np.load(max0_path + 'losses.npy'), np.load(max0_path + 'test_risk.npy')
                eff_gains, rcp_lams = np.load(max0_path + 'eff_gains.npy'), np.load(max0_path + 'rcp_lams.npy', allow_pickle=True)
                plot_efficiency_gains(ax, eps_grid, eff_gains, label + ' clipped', color, 'dashed', n_early_exit)
                # Scaling
                scaling_path = precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/' + label + '/'
                losses, test_risk = np.load(scaling_path + 'losses.npy'), np.load(scaling_path + 'test_risk.npy')
                eff_gains, rcp_lams = np.load(scaling_path + 'eff_gains.npy'), np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
                plot_efficiency_gains(ax, eps_grid, eff_gains, label + ' with transformation', color, 'solid', n_early_exit)

            if display_legends:
                ax.legend()
            
            plt.tight_layout()
            
            if debug_mode:
                # Display the image
                plt.show()
            else:
                # Save out the image
                path = plot_directory + 'eff_gains_with_comparison/'
                if not os.path.exists(path):
                    os.makedirs(path)
                plt.savefig(path + dataset + '_' + model_name.split('/')[1] + '.pdf')
    
                # Close plots to save memory
                matplotlib.pyplot.close()
        else:
            print('Missing data: ', dataset, model_name)

    print('Finished', dataset)

Finished sst2
Finished trec
Missing data:  financial_phrasebank Qwen/Qwen3-8B
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Accuracy vs Layer
Paper: fig 1, 15, 17, 19

In [58]:
# Accuracy vs Layer plots (showing overthinking)
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        fig, ax = plt.subplots(1,1,figsize=(5,5))
        label_order = get_label_order(dataset, tokenizer)
        base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
        
        # First check that there exists all types of experiments
        if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                and os.path.exists(base_dir + 'zeroshot.json')):
            # We have all the data
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)
            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            
            for gt, label, color in zip([c_gt, i_gt, z_gt], ['correct', 'incorrect', 'zeroshot'], [c_color, i_color, z_color]):
                acc = get_all_accuracies(data[label], gt, n_early_exit, 0)
                ax.plot([i for i in range(n_early_exit)], acc.mean(axis=0), label=label, color=color)

            if display_legends:
                ax.legend()
            
            ax.set_xlabel('Layer')
            calib = "Calibrated" if use_calibration else "Uncalibrated"
            ax.set_ylabel(calib + ' Accuracy')
            plt.tight_layout()
            if debug_mode:
                # Display the image
                plt.show()
            else:
                # Save out the image
                path = plot_directory + 'loss_vs_layer/' 
                if not os.path.exists(path):
                    os.makedirs(path)
                plt.savefig(path + dataset + '_' + model_name.split('/')[1] + '.pdf')
        
                # Close plots to save memory
                matplotlib.pyplot.close()
        else:
            print('Missing data: ', dataset, model_name)
    
    print('Finished', dataset)

Finished sst2
Finished trec
Missing data:  financial_phrasebank Qwen/Qwen3-8B
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Lambda vs Risk - Correct vs Incorrect Demos and Scaling vs Clipping
Paper: fig 18, top row.

In [60]:
# Plot lambda vs risk for both risk types, correct and incorrect demos
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        fig, ax = plt.subplots(1,1,figsize=(5,5))
        label_order = get_label_order(dataset, tokenizer)
        if fake_labels:
            label_order = [fake_label_map[x] for x in label_order]
        
        base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
        # First check that there exists all types of experiments
        if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                and os.path.exists(base_dir + 'zeroshot.json')):
            # We have all the data
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

            # Plot lambda vs correct risk, scaled
            conf = get_all_confidences(data['correct'], n_early_exit, label_order, confidence_type, int(n_early_exit/2))
            acc = get_all_accuracies(data['correct'], c_gt, n_early_exit, int(n_early_exit/2))
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, c_rel, c_gt)
            c_risk = losses.mean(axis=1)
            ax.plot(lambdas, c_risk, label='correct with scaling', color=c_color)

            # Plot lambda vs correct risk, clipped
            losses = losses.clip(min=0)
            c_risk = losses.mean(axis=1)
            ax.plot(lambdas, c_risk, label='correct with clipping', color=c_color, linestyle='dashed')

            # Plot lambda vs incorrect risk, scaled
            conf = get_all_confidences(data['incorrect'], n_early_exit, label_order, confidence_type, int(n_early_exit/2))
            acc = get_all_accuracies(data['incorrect'], c_gt, n_early_exit, int(n_early_exit/2))
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, i_rel, i_gt)
            i_risk = losses.mean(axis=1)
            ax.plot(lambdas, i_risk, label='incorrect with scaling', color=i_color)

            # Plot lambda vs incorrect risk, scaled
            losses = losses.clip(min=0)
            i_risk = losses.mean(axis=1)
            ax.plot(lambdas, i_risk, label='incorrect with clipping', color=i_color, linestyle='dashed')
            
            ax.set_xlabel('Lambda')
            ax.set_ylabel('Empirical Risk')
            if display_legends:
                ax.legend()
                
            plt.tight_layout()
            
            if debug_mode:
                # Display the image
                plt.show()
            else:
                # Save out the image
                path = plot_directory + mixed_filename + '/lambda_vs_risk/'
                if not os.path.exists(path):
                    os.makedirs(path)
                plt.savefig(path + dataset + '_' + model_name.split('/')[1] + '.pdf')
    
                # Close plots to save memory
                matplotlib.pyplot.close()
        else:
            print('Missing data: ', dataset, model_name)
    
    print('Finished', dataset)

Finished sst2
Finished trec
Missing data:  financial_phrasebank Qwen/Qwen3-8B
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Confidence vs Layer
Paper: fig 19

In [62]:
# Plot confidence of each layer's prediction through layers
for dataset in datasets:
    for model_idx in range(len(models)):
        fig, ax = plt.subplots(1,1,figsize=(5,5))
        model_name, n_early_exit, tokenizer = models[model_idx], n_early_exits[model_idx], tokenizers[model_idx]
        layers = [i for i in range(n_early_exit)]
        label_order = get_label_order(dataset, tokenizer)
        if fake_labels:
            label_order = [fake_label_map[x] for x in label_order]
        
        base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
        # First check that there exists all types of experiments
        if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                and os.path.exists(base_dir + 'zeroshot.json')):
            # We have all the data
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_conf = get_all_confidences(data['correct'], n_early_exit, label_order, confidence_type)
            i_conf = get_all_confidences(data['incorrect'], n_early_exit, label_order, confidence_type)
            z_conf = get_all_confidences(data['zeroshot'], n_early_exit, label_order, confidence_type)

            ax.plot(layers, c_conf.mean(axis=0), label='correct')
            ax.plot(layers, i_conf.mean(axis=0), label='incorrect')
            ax.plot(layers, z_conf.mean(axis=0), label='zeroshot')
            ax.set_xlabel('Layer')
            ax.set_ylabel('Confidence')
            if display_legends:
                ax.legend()

            plt.tight_layout()

        if debug_mode:
            # Display the image
            plt.show()
        else:
            # Save out the image
            path = plot_directory + 'confidences/' + model_name + '/'
            if not os.path.exists(path):
                os.makedirs(path)
            plt.savefig(path + dataset + '.pdf')

    # Close plots to save memory
    matplotlib.pyplot.close()
    print('Finished', dataset)

Finished sst2
Finished trec
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Comparing Accuracy vs Full Model (instead of zero-shot) - Table
Not in paper; used only for rebuttals. 

In [64]:
# Print out data from the separate risk-control data
eps = 0.2
eps_index = np.where(eps_grid == eps)[0][0]
print('Using epsilon value', eps)

# Convert to Markdown format
def to_markdown_table(rows):
    header = "| " + " | ".join(rows[0]) + " |"
    separator = "| " + " | ".join(["---"] * len(rows[0])) + " |"
    content = "\n".join("| " + " | ".join(row) + " |" for row in rows[1:])
    return "\n".join([header, separator, content])

inc_rows = [["Dataset", "Zeroshot Performance", "Full Model Accuracy", "Early Exit Accuracy", "Accuracy Gain vs Full Model",
                "Efficiency Gain vs Full Model"]]
correct_rows = [["Dataset", "Zeroshot Performance", "Full Model Accuracy", "Early Exit Accuracy", "Accuracy Gain vs Full Model",
                "Efficiency Gain vs Full Model"]]
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        if 'layerskip-llama3-8B' in model_name:
            print('Running', dataset, model_name)
            # First check that all experiment results are precomputed
            if os.path.exists(precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/'):
                label_order = get_label_order(dataset, tokenizer)
                if fake_labels:
                    label_order = [fake_label_map[x] for x in label_order]
                
                # load the data
                base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
                data = {}
                for expt_type in ['correct', 'incorrect', 'zeroshot']:
                    with open(base_dir + expt_type + '.json', 'r') as file:
                        data[expt_type] = json.load(file)
    
                c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
                c_rel, i_rel, z_rel = get_relative_labels("zeroshot_full_model", data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
                
                # For each dataset, print the following (TODO averaged over all models):
                # Zeroshot, incorrect, correct demos full-model performance
                c_acc = np.mean(get_all_accuracies(data['correct'], c_gt, n_early_exit, int(n_early_exit/2))[:,-1])
                i_acc = np.mean(get_all_accuracies(data['incorrect'], i_gt, n_early_exit, int(n_early_exit/2))[:,-1])
                z_acc = np.mean(get_all_accuracies(data['zeroshot'], z_gt, n_early_exit, int(n_early_exit/2))[:,-1])
                #print('Zeroshot acc:', z_acc)
                #print('Correct full model:', c_acc)
                #print('Incorrect full model:', i_acc)

                # Set default loss/eff gain when risk can't be controlled
                def_loss_uncontrolled_risk = 1-z_acc 
                def_eff_gain_uncontrolled_risk = 0

                # Compute early exit acc, delta w full model, and eff gains for correct demos
                conf = get_all_confidences(data['correct'], n_early_exit, label_order, confidence_type, int(n_early_exit/2))
                acc = get_all_accuracies(data['correct'], c_gt, n_early_exit, int(n_early_exit/2))
                n_cal = int(len(data['correct']['0'])/2)
                losses, test_risk, eff_gains, rcp_lams = apply_risk_control(conf, acc, c_gt, c_rel, lambdas, eps_grid,
                                                                            delta, n_cal, n_trials, 'scaling')
                losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, None, c_gt, False)
                # Get the loss corresponding to the right lambda
                if rcp_lams[eps_index] is not None:
                    lam_idx = rcp_lams[eps_index]
                    c_acc_ee = 1-losses[lam_idx,:].mean(axis=0)
                else:
                    c_acc_ee = z_acc
                    c_eff_gain = 0
                #print('Early exit correct acc:', c_acc_ee)
                #print('Delta w full model:', c_acc-c_acc_ee)
                c_eff_gain = eff_gains.mean(axis=0)[eps_index]
                #print('Efficiency gain correct:', c_eff_gain)
    
                # Compute early exit acc, delta w full model, and eff gains for incorrect demos
                conf = get_all_confidences(data['incorrect'], n_early_exit, label_order, confidence_type, int(n_early_exit/2))
                acc = get_all_accuracies(data['incorrect'], i_gt, n_early_exit, int(n_early_exit/2))
                n_cal = int(len(data['incorrect']['0'])/2)
                losses, test_risk, eff_gains, rcp_lams = apply_risk_control(conf, acc, i_gt, i_rel, lambdas, eps_grid, 
                                                                            delta, n_cal, n_trials, 'scaling')
                losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, None, i_gt, False)
                # Get the loss corresponding to the right lambda
                if rcp_lams[eps_index] is not None:
                    lam_idx = rcp_lams[eps_index]
                    i_acc_ee = 1-losses[lam_idx,:].mean(axis=0)
                else:
                    i_acc_ee = z_acc
                    i_eff_gain = 0
                #print('Early exit incorrect acc:', i_acc_ee)
                #print('Delta w full model:', i_acc-i_acc_ee)
                i_eff_gain = eff_gains.mean(axis=0)[eps_index]
                #print('Efficiency gain incorrect:', i_eff_gain)
                c_results = [dataset, round(z_acc,3), round(c_acc,3), round(c_acc_ee,3), round(c_acc-c_acc_ee,3), round(c_eff_gain,3)]
                correct_rows.append([str(x) for x in c_results])
                i_results = [dataset, round(z_acc,3), round(i_acc,3), round(i_acc_ee,3), round(i_acc-i_acc_ee,3), round(i_eff_gain,3)]
                inc_rows.append([str(x) for x in i_results])
                # print('Correct:', round(z_acc,3), round(c_acc,3), round(c_acc_ee,3), round(c_acc-c_acc_ee,3), round(c_eff_gain,3))
                # print('Incorrect:', round(z_acc,3), round(i_acc,3), round(i_acc_ee,3), round(i_acc-i_acc_ee,3), round(i_eff_gain,3))

print(to_markdown_table(correct_rows), '\n\n')
print(to_markdown_table(inc_rows))

Using epsilon value 0.2
Running sst2 facebook/layerskip-llama3-8B
Running trec facebook/layerskip-llama3-8B
Running financial_phrasebank facebook/layerskip-llama3-8B
Running tweeteval_hate facebook/layerskip-llama3-8B
Running tweeteval_feminist facebook/layerskip-llama3-8B
Running tweeteval_atheism facebook/layerskip-llama3-8B
Running unnatural facebook/layerskip-llama3-8B
Running ag_news facebook/layerskip-llama3-8B
| Dataset | Zeroshot Performance | Full Model Accuracy | Early Exit Accuracy | Accuracy Gain vs Full Model | Efficiency Gain vs Full Model |
| --- | --- | --- | --- | --- | --- |
| sst2 | 0.818 | 0.929 | 0.811 | 0.119 | 1.0 |
| trec | 0.275 | 0.568 | 0.254 | 0.314 | 1.0 |
| financial_phrasebank | 0.501 | 0.712 | 0.476 | 0.236 | 1.0 |
| tweeteval_hate | 0.547 | 0.637 | 0.602 | 0.035 | 1.0 |
| tweeteval_feminist | 0.345 | 0.631 | 0.49 | 0.142 | 1.0 |
| tweeteval_atheism | 0.43 | 0.695 | 0.542 | 0.153 | 1.0 |
| unnatural | 0.756 | 1.0 | 0.978 | 0.022 | 1.0 |
| ag_news | 0.306